# 03 - Data Structures and Algorithms (Java)

This notebook builds the same fault-stack / action-queue / CAN-ID-lookup example from `concept.md`, idiomatically in Java, then runs the same Big-O demo comparing linear search against a hashmap lookup. Read `concept.md` first if you haven't.

## Imports

In [1]:
import java.util.ArrayDeque;
import java.util.Deque;
import java.util.HashMap;
import java.util.List;
import java.util.ArrayList;
import java.util.Map;

## Stack: Recent Controller Faults

Java's `Deque` interface (via the `ArrayDeque` implementation) covers both stacks and queues — Java doesn't have a separate dedicated `Stack` type worth using today (the old `java.util.Stack` class exists but is legacy and slower). `push()`/`pop()` on a `Deque` operate on the front, giving Last In, First Out.

In [2]:
Deque<String> faultStack = new ArrayDeque<>();

faultStack.push("CAN timeout: device 12");
faultStack.push("Brownout detected");
faultStack.push("CAN timeout: device 7");

System.out.println("Most recent fault first:");
while (!faultStack.isEmpty()) {
    System.out.println(" - " + faultStack.pop());
}

Most recent fault first:


 - CAN timeout: device 7


 - Brownout detected


 - CAN timeout: device 12


Same result as the Python notebook: the last fault pushed ("CAN timeout: device 7") is the first one popped.

## Queue: Autonomous Action Sequence

The exact same `ArrayDeque` object can be used as a queue instead of a stack, just by calling different methods: `offer()` to enqueue at the back, `poll()` to dequeue from the front.

In [3]:
Deque<String> actionQueue = new ArrayDeque<>();
actionQueue.offer("drive forward");
actionQueue.offer("intake");
actionQueue.offer("shoot");
actionQueue.offer("drive back");

System.out.println("Executing in queued order:");
while (!actionQueue.isEmpty()) {
    System.out.println(" - " + actionQueue.poll());
}

Executing in queued order:


 - drive forward


 - intake


 - shoot


 - drive back


Same underlying class (`ArrayDeque`), completely different behavior, purely because of which methods we called — a good reminder that "stack" and "queue" are really about *which operations you use*, not some fundamentally different piece of hardware.

## Hashmap: CAN ID to Device Name

Java's `HashMap` is the direct equivalent of Python's `dict`: key-value pairs, average O(1) lookup by key.

In [4]:
Map<Integer, String> canIdToName = new HashMap<>();
canIdToName.put(1, "Front Left Drive");
canIdToName.put(2, "Front Right Drive");
canIdToName.put(3, "Back Left Drive");
canIdToName.put(4, "Back Right Drive");
canIdToName.put(12, "Intake Roller");

System.out.println(canIdToName.get(12));
System.out.println(canIdToName.getOrDefault(99, "<unknown device>"));

Intake Roller


<unknown device>


`getOrDefault(key, default)` is Java's equivalent of Python's `.get(key, default)` — a safe lookup that doesn't throw or return `null` for a missing key.

## Big-O in Practice: Linear Search vs. Hashmap Lookup

Same experiment as the Python notebook: a big table of (id, name) pairs, and a count of how many comparisons a linear scan needs to find an ID at different positions.

In [5]:
int linearSearchComparisons(List<int[]> entries, int targetId) {
    int comparisons = 0;
    for (int[] entry : entries) {
        comparisons++;
        if (entry[0] == targetId) {
            return comparisons;
        }
    }
    return comparisons;
}

List<int[]> bigTable = new ArrayList<>();
Map<Integer, String> bigMap = new HashMap<>();
for (int i = 0; i < 10_000; i++) {
    bigTable.add(new int[] { i });
    bigMap.put(i, "device-" + i);
}

for (int target : new int[] {5, 5_000, 9_999}) {
    int comparisons = linearSearchComparisons(bigTable, target);
    System.out.println("linear search for id=" + target + ": " + comparisons + " comparisons");
}

System.out.println();
System.out.println("hashmap lookup for id=5:    " + bigMap.get(5));
System.out.println("hashmap lookup for id=9999: " + bigMap.get(9999));

linear search for id=5: 6 comparisons


linear search for id=5000: 5001 comparisons


linear search for id=9999: 10000 comparisons


hashmap lookup for id=5:    device-5


hashmap lookup for id=9999: device-9999


Same conclusion as the Python notebook: the comparison count for linear search scales directly with the target's position in the table, while the hashmap lookups do roughly the same amount of work regardless of where the entry actually is.

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor.

1. Add a `peek()` call to the fault stack example that looks at the most recent fault *without* removing it.
2. Modify `linearSearchComparisons` to also work correctly for a target that isn't in `bigTable` at all, and confirm it returns `bigTable.size()`.
3. Time (with `System.nanoTime()`) an actual `bigMap.get(9999)` versus an actual `linearSearchComparisons(bigTable, 9999)` call, over many repetitions, and see if the wall-clock gap matches what the comparison counts predicted.

In [6]:
// Your code here


## Resources

- [Oracle Java Tutorials: The Collections Framework](https://docs.oracle.com/javase/tutorial/collections/index.html) - `Deque`, `HashMap`, and the rest of Java's built-in data structures.
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/) - time/space complexity for common data structures and algorithms.
- [WPILib `SequentialCommandGroup`](https://docs.wpilib.org/en/stable/docs/software/commandbased/command-groups.html) - the real queue-like structure behind chained autonomous actions on the robot.